# Questions for Nick

- SAT and SBT - make summary table for nick, nick email JC
- SUP - are meds correct? - meds sent to Nick
- ECMO - where is the raw file? make clif table
- alcohol withdrawl, seizure, increased icp - icd10 code check

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import math
from pandas.api.types import (
    is_numeric_dtype, is_bool_dtype, is_categorical_dtype,
    is_datetime64_any_dtype
)
import duckdb
import subprocess
import sys
import textwrap
import json
import os
import shutil

# Global Settings

In [ ]:
# create intermediate_outputs if needed
output_path = 'intermediate_outputs'
os.makedirs(output_path, exist_ok=True)

pd.set_option('display.max_columns', None)

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["paths"]["clif"]
raw_path = cfg["paths"]["raw"]
db_path = cfg["paths"]["db"]
timezone = cfg['R_timezone']

con = duckdb.connect(database=f'{output_path}/ltvv.duckdb')
con.execute(f"SET TimeZone = '{timezone}'")

print(f"File path: {clif_path}")
print(f"Time zone: {timezone}")

# File paths

In [ ]:
hourly_resp_path = f'{db_path}/clif_hourly_resp_support.parquet'
patient_path = f'{clif_path}/clif_patient.parquet'
hosp_path = f'{clif_path}/clif_hospitalization.parquet'
vitals_path = f"{clif_path}/clif_vitals.parquet"
provider_path = f'{clif_path}/clif_provider.parquet'
position_path = f'{clif_path}/clif_position.parquet'
meds_continuous_path = f'{clif_path}/clif_medication_admin_continuous.parquet'
meds_intermittent_path = f'{clif_path}/clif_medication_admin_intermittent.parquet'
labs_path = f'{clif_path}/clif_labs.parquet'
micro_non_culture_path = f'{clif_path}/clif_microbiology_nonculture.parquet'
ecmo_path = f'{clif_path}/clif_ecmo_mcs.parquet'
diag_path = f"{clif_path}/clif_admission_diagnosis.parquet"
resp_supp_path = f"{clif_path}/clif_respiratory_support.parquet"
adt_path = f'{clif_path}/clif_adt.parquet'

# Get starter vent cohort

## Get cohort metadata from hourly table

In [ ]:
# The hourly table's only role is to identify qualifying hospitalizations
# (vent episode 1, duration >= 24h, ICU location) and supply metadata that
# is not present in the raw respiratory_support table: ibw, hospital_id,
# episode duration, and the episode-1 end timestamp used to bound all
# downstream raw-table queries so episode 2+ records are excluded.

con.sql(f"""
CREATE OR REPLACE TEMP TABLE cohort_meta AS
WITH ep1 AS (
    SELECT
        hospitalizations_joined_id,
        ANY_VALUE(ibw)  FILTER (WHERE ibw IS NOT NULL) AS ibw,
        ANY_VALUE(hospital_id)                         AS hospital_id,
        MAX(vent_episode_duration_hours)               AS vent_episode_duration_hours,
        -- Naive local-time end of episode 1; used to filter raw records to ep1 only
        MAX(
            MAKE_TIMESTAMP(
                YEAR(recorded_date), MONTH(recorded_date), DAY(recorded_date),
                CAST(recorded_hour AS INT), 59, 59
            )
        ) AS ep1_end_local
    FROM '{hourly_resp_path}'
    WHERE device_category = 'imv'
        AND vent_episode_id = '1'
        AND vent_episode_duration_hours >= 24
        AND location_category = 'icu'
    GROUP BY hospitalizations_joined_id
),
patient_ids AS (
    SELECT hospitalizations_joined_id, ANY_VALUE(patient_id) AS mdm_link_id
    FROM '{hosp_path}'
    GROUP BY hospitalizations_joined_id
)
SELECT ep1.*, patient_ids.mdm_link_id
FROM ep1
LEFT JOIN patient_ids USING (hospitalizations_joined_id);

SELECT COUNT(*) AS n_cohort FROM cohort_meta
""")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE data AS
WITH
-- Hourly-structured provider table; join at (date, hour) derived from raw timestamps
provider_data AS (
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_date)        AS date,
        CAST(recorded_hour AS INT) AS recorded_hour,
        prov_npi_shifted,
        prov_name_shifted,
        prov_spec_shifted
    FROM '{provider_path}'
),
-- First IMV record per cohort hospitalization within episode 1 = intubation proxy
intubation_times AS (
    SELECT
        rs.hospitalizations_joined_id,
        MIN(rs.recorded_dttm) AS intubation_dttm
    FROM '{resp_supp_path}' rs
    INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    GROUP BY rs.hospitalizations_joined_id
),
-- Day-1: first non-null tidal_volume_set >= 5h post-intubation within episode 1.
-- QUALIFY ROW_NUMBER() = 1 selects the full row at the earliest qualifying record,
-- guaranteeing all columns come from the same raw charting event.
day1_recs AS (
    SELECT rs.*
    FROM '{resp_supp_path}' rs
    INNER JOIN intubation_times it USING (hospitalizations_joined_id)
    INNER JOIN cohort_meta cm       USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.tidal_volume_set IS NOT NULL
      AND rs.recorded_dttm >= it.intubation_dttm + INTERVAL 5 HOURS
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY rs.hospitalizations_joined_id
        ORDER BY rs.recorded_dttm
    ) = 1
),
-- Day-1 local date (to exclude from subseq and tag initial_vent_day)
day1_dates AS (
    SELECT
        hospitalizations_joined_id,
        CAST(recorded_dttm AT TIME ZONE '{timezone}' AS DATE) AS day1_date
    FROM day1_recs
),
-- Subsequent days: first non-null tidal_volume_set >= 14:00 local per calendar date,
-- within episode 1, excluding day1_date.
-- Only dates where Vt was actually charted appear (no phantom days).
subseq_recs AS (
    SELECT rs.*
    FROM '{resp_supp_path}' rs
    INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
    INNER JOIN day1_dates d1  USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.tidal_volume_set IS NOT NULL
      AND EXTRACT(HOUR FROM rs.recorded_dttm AT TIME ZONE '{timezone}') >= 14
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) != d1.day1_date
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY rs.hospitalizations_joined_id,
                     CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE)
        ORDER BY rs.recorded_dttm
    ) = 1
),
-- Stack day-1 and subsequent records; derive date and recorded_hour from raw timestamps
all_reps AS (
    SELECT
        hospitalizations_joined_id,
        recorded_dttm,
        CAST(recorded_dttm AT TIME ZONE '{timezone}' AS DATE)             AS date,
        CAST(EXTRACT(HOUR FROM recorded_dttm AT TIME ZONE '{timezone}') AS INT) AS recorded_hour,
        tidal_volume_set,
        tidal_volume_obs,
        fio2_set,
        peep_set,
        tracheostomy,
        mode_category AS vent_mode_category,
        TRUE          AS initial_vent_day
    FROM day1_recs
    UNION ALL
    SELECT
        hospitalizations_joined_id,
        recorded_dttm,
        CAST(recorded_dttm AT TIME ZONE '{timezone}' AS DATE)             AS date,
        CAST(EXTRACT(HOUR FROM recorded_dttm AT TIME ZONE '{timezone}') AS INT) AS recorded_hour,
        tidal_volume_set,
        tidal_volume_obs,
        fio2_set,
        peep_set,
        tracheostomy,
        mode_category AS vent_mode_category,
        FALSE         AS initial_vent_day
    FROM subseq_recs
),
-- Join provider at (date, recorded_hour) derived from raw timestamp
reps_with_prov AS (
    SELECT
        a.*,
        p.prov_npi_shifted,
        p.prov_name_shifted,
        p.prov_spec_shifted
    FROM all_reps a
    LEFT JOIN provider_data p USING (hospitalizations_joined_id, date, recorded_hour)
),
-- Provider threshold: >= 25 day-1 encounters
provider_counts AS (
    SELECT prov_npi_shifted, COUNT(*) AS count
    FROM reps_with_prov
    WHERE initial_vent_day = TRUE
      AND prov_npi_shifted IS NOT NULL
    GROUP BY prov_npi_shifted
    HAVING count >= 25
)
-- Final: qualifying providers only, joined with cohort metadata
SELECT
    r.hospitalizations_joined_id,
    r.recorded_dttm,
    r.date,
    r.recorded_hour,
    r.tidal_volume_set,
    r.tidal_volume_obs,
    r.fio2_set,
    r.peep_set,
    r.tracheostomy,
    r.vent_mode_category,
    r.initial_vent_day,
    r.prov_npi_shifted,
    r.prov_name_shifted,
    r.prov_spec_shifted,
    cm.ibw,
    cm.hospital_id,
    cm.vent_episode_duration_hours,
    cm.mdm_link_id,
    YEAR(r.date) AS recorded_year
FROM reps_with_prov r
INNER JOIN provider_counts pc USING (prov_npi_shifted)
INNER JOIN cohort_meta cm      USING (hospitalizations_joined_id)
ORDER BY hospitalizations_joined_id, date, recorded_hour;

SELECT COUNT(*) AS n_rows, COUNT(DISTINCT prov_npi_shifted) AS n_providers FROM data
""")

In [ ]:
# Task 20 — Figure 1 Provider Drop-Off Explanation (R3 Minor #11)
# Re-runs the minimal CTE chain from Cell 11 (up to reps_with_prov) to count
# providers before and after the >=25 day-1 episode threshold, then
# characterises the excluded pool so Claire can explain the 10:1 drop-off.

# Materialise episode counts once; query twice (summary + buckets)
con.sql(f"""
CREATE OR REPLACE TEMP TABLE task20_episode_counts AS
WITH
provider_data AS (
    SELECT hospitalizations_joined_id,
           DATE(recorded_date)        AS date,
           CAST(recorded_hour AS INT) AS recorded_hour,
           prov_npi_shifted
    FROM '{provider_path}'
),
intubation_times AS (
    SELECT rs.hospitalizations_joined_id, MIN(rs.recorded_dttm) AS intubation_dttm
    FROM '{resp_supp_path}' rs
    INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    GROUP BY rs.hospitalizations_joined_id
),
day1_recs AS (
    SELECT rs.hospitalizations_joined_id, rs.recorded_dttm
    FROM '{resp_supp_path}' rs
    INNER JOIN intubation_times it USING (hospitalizations_joined_id)
    INNER JOIN cohort_meta cm       USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.tidal_volume_set IS NOT NULL
      AND rs.recorded_dttm >= it.intubation_dttm + INTERVAL 5 HOURS
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY rs.hospitalizations_joined_id ORDER BY rs.recorded_dttm
    ) = 1
),
day1_with_hour AS (
    SELECT hospitalizations_joined_id,
           CAST(recorded_dttm AT TIME ZONE '{timezone}' AS DATE)             AS date,
           CAST(EXTRACT(HOUR FROM recorded_dttm AT TIME ZONE '{timezone}') AS INT) AS recorded_hour
    FROM day1_recs
),
day1_with_prov AS (
    SELECT d.hospitalizations_joined_id, p.prov_npi_shifted
    FROM day1_with_hour d
    LEFT JOIN provider_data p USING (hospitalizations_joined_id, date, recorded_hour)
    WHERE p.prov_npi_shifted IS NOT NULL
)
SELECT prov_npi_shifted, COUNT(*) AS n_episodes
FROM day1_with_prov
GROUP BY prov_npi_shifted
""")

# --- Summary stats ---
provider_dropoff_stats = con.sql("""
SELECT
    COUNT(*)                                                                      AS n_providers_total,
    SUM(CASE WHEN n_episodes >= 25 THEN 1 ELSE 0 END)                           AS n_providers_included,
    SUM(CASE WHEN n_episodes <  25 THEN 1 ELSE 0 END)                           AS n_providers_excluded,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY n_episodes)
        FILTER (WHERE n_episodes < 25)                                            AS median_eps_excluded,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY n_episodes)
        FILTER (WHERE n_episodes < 25)                                            AS q1_eps_excluded,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY n_episodes)
        FILTER (WHERE n_episodes < 25)                                            AS q3_eps_excluded,
    SUM(CASE WHEN n_episodes < 5 THEN 1 ELSE 0 END)                             AS n_excluded_lt5,
    ROUND(
        SUM(CASE WHEN n_episodes < 5 THEN 1.0 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN n_episodes < 25 THEN 1 ELSE 0 END), 0) * 100, 1
    )                                                                             AS pct_excluded_lt5
FROM task20_episode_counts
""").df()

print('=== Task 20: Provider Drop-Off Summary ===')
display(provider_dropoff_stats)

# --- Bucketed frequency table of excluded providers ---
provider_dropoff_buckets = con.sql("""
SELECT
    CASE
        WHEN n_episodes = 1              THEN '1'
        WHEN n_episodes BETWEEN 2 AND  4 THEN '2-4'
        WHEN n_episodes BETWEEN 5 AND  9 THEN '5-9'
        WHEN n_episodes BETWEEN 10 AND 14 THEN '10-14'
        WHEN n_episodes BETWEEN 15 AND 19 THEN '15-19'
        WHEN n_episodes BETWEEN 20 AND 24 THEN '20-24'
    END                              AS episode_bucket,
    COUNT(*)                         AS n_providers,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_excluded
FROM task20_episode_counts
WHERE n_episodes < 25
GROUP BY episode_bucket
ORDER BY MIN(n_episodes)
""").df()

print('\n=== Task 20: Episode Distribution of Excluded Providers ===')
display(provider_dropoff_buckets)

# --- Save plain-text summary ---
s = provider_dropoff_stats.iloc[0]
summary_lines = [
    'Task 20 - Figure 1 Provider Drop-Off Explanation',
    '=' * 50,
    f'Pre-threshold provider pool (>=1 day-1 episode with non-null NPI): {int(s["n_providers_total"]):,}',
    f'Included (>=25 day-1 episodes):  {int(s["n_providers_included"]):,}',
    f'Excluded (<25 day-1 episodes):   {int(s["n_providers_excluded"]):,}',
    f'Excluded providers - median episodes: {s["median_eps_excluded"]:.0f} [IQR {s["q1_eps_excluded"]:.0f}-{s["q3_eps_excluded"]:.0f}]',
    f'Excluded providers with <5 episodes: {int(s["n_excluded_lt5"]):,} ({s["pct_excluded_lt5"]:.1f}%)',
    '',
    'Episode bucket distribution (excluded providers):',
]
for _, row in provider_dropoff_buckets.iterrows():
    summary_lines.append(f"  {row['episode_bucket']:>5} episodes: {int(row['n_providers']):>4} providers ({row['pct_of_excluded']:.1f}%)")

summary_text = '\n'.join(summary_lines) + '\n'
with open(f'{output_path}/task20_provider_dropoff.txt', 'w') as fh:
    fh.write(summary_text)
print('\n' + summary_text)


# Non-VC Mode Exclusion Count (Task 14)

In [ ]:
# Task 14 — Non-VC Mode Exclusion Count (R2 Minor #3)
# Compares measurement-window IMV rows with/without tidal_volume_set to quantify
# how many patients (day-1) and patient-days (subsequent-day) were excluded.
# Among excluded rows, mode_category separates true non-VC modes from VC data gaps.

# Shared CTEs: intubation proxy and day-1 date anchor (Vt-not-null, same as main analysis)
_shared_ctes = f"""
    intubation_times AS (
        SELECT
            rs.hospitalizations_joined_id,
            MIN(rs.recorded_dttm) AS intubation_dttm
        FROM '{resp_supp_path}' rs
        INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
        WHERE rs.device_category ILIKE '%imv%'
          AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
                  <= cm.ep1_end_local + INTERVAL 2 HOURS
        GROUP BY rs.hospitalizations_joined_id
    ),
    day1_date_anchor AS (
        SELECT
            rs.hospitalizations_joined_id,
            CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) AS day1_date
        FROM '{resp_supp_path}' rs
        INNER JOIN intubation_times it USING (hospitalizations_joined_id)
        INNER JOIN cohort_meta cm       USING (hospitalizations_joined_id)
        WHERE rs.device_category ILIKE '%imv%'
          AND rs.tidal_volume_set IS NOT NULL
          AND rs.recorded_dttm >= it.intubation_dttm + INTERVAL 5 HOURS
          AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
                  <= cm.ep1_end_local + INTERVAL 2 HOURS
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY rs.hospitalizations_joined_id
            ORDER BY rs.recorded_dttm
        ) = 1
    )
"""

# --- Query 1: included vs excluded summary per stratum ---
summary_14 = con.sql(f"""
WITH
{_shared_ctes},
day1_all AS (
    SELECT rs.tidal_volume_set, rs.mode_category
    FROM '{resp_supp_path}' rs
    INNER JOIN intubation_times it USING (hospitalizations_joined_id)
    INNER JOIN cohort_meta cm       USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.recorded_dttm >= it.intubation_dttm + INTERVAL 5 HOURS
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY rs.hospitalizations_joined_id
        ORDER BY rs.recorded_dttm
    ) = 1
),
subseq_all AS (
    SELECT rs.tidal_volume_set, rs.mode_category
    FROM '{resp_supp_path}' rs
    INNER JOIN cohort_meta cm       USING (hospitalizations_joined_id)
    INNER JOIN day1_date_anchor da  USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND EXTRACT(HOUR FROM rs.recorded_dttm AT TIME ZONE '{timezone}') >= 14
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) != da.day1_date
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY rs.hospitalizations_joined_id,
                     CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE)
        ORDER BY rs.recorded_dttm
    ) = 1
)
SELECT
    'Day-1 (patients)'             AS stratum,
    COUNT(*)                       AS n_total,
    SUM(CASE WHEN tidal_volume_set IS NOT NULL THEN 1 ELSE 0 END) AS n_included,
    SUM(CASE WHEN tidal_volume_set IS NULL     THEN 1 ELSE 0 END) AS n_excluded,
    ROUND(SUM(CASE WHEN tidal_volume_set IS NULL THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS pct_excluded
FROM day1_all
UNION ALL
SELECT
    'Subsequent-day (patient-days)' AS stratum,
    COUNT(*),
    SUM(CASE WHEN tidal_volume_set IS NOT NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN tidal_volume_set IS NULL     THEN 1 ELSE 0 END),
    ROUND(SUM(CASE WHEN tidal_volume_set IS NULL THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1)
FROM subseq_all
""").df()

print("=== Task 14: Exclusion Summary ===")
display(summary_14)

# --- Query 2: mode_category breakdown among excluded rows ---
mode_14 = con.sql(f"""
WITH
{_shared_ctes},
day1_excluded AS (
    SELECT 'Day-1 (patients)' AS stratum, rs.mode_category
    FROM '{resp_supp_path}' rs
    INNER JOIN intubation_times it USING (hospitalizations_joined_id)
    INNER JOIN cohort_meta cm       USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.tidal_volume_set IS NULL
      AND rs.recorded_dttm >= it.intubation_dttm + INTERVAL 5 HOURS
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY rs.hospitalizations_joined_id
        ORDER BY rs.recorded_dttm
    ) = 1
),
subseq_excluded AS (
    SELECT 'Subsequent-day (patient-days)' AS stratum, rs.mode_category
    FROM '{resp_supp_path}' rs
    INNER JOIN cohort_meta cm       USING (hospitalizations_joined_id)
    INNER JOIN day1_date_anchor da  USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND rs.tidal_volume_set IS NULL
      AND EXTRACT(HOUR FROM rs.recorded_dttm AT TIME ZONE '{timezone}') >= 14
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) != da.day1_date
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY rs.hospitalizations_joined_id,
                     CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE)
        ORDER BY rs.recorded_dttm
    ) = 1
)
SELECT
    stratum,
    COALESCE(mode_category, 'Unknown / NULL') AS mode_category,
    COUNT(*) AS n,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY stratum), 1) AS pct_of_excluded
FROM (SELECT * FROM day1_excluded UNION ALL SELECT * FROM subseq_excluded)
GROUP BY stratum, mode_category
ORDER BY stratum, n DESC
""").df()

print("\n=== Task 14: Mode Category of Excluded Rows ===")
print("(Non-VC modes = true mode exclusion; volume_control with NULL Vt = data gap)")
display(mode_14)

In [ ]:
con.sql(f"SELECT * FROM data")

# Patient

In [ ]:
con.sql(f"SELECT * FROM '{patient_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW patient AS
    SELECT DISTINCT ON (mdm_link_id)
        mdm_link_id,
        sex_category,
        language_category,
        race_category,
        ethnicity_category,
        death_dttm,
        DATE(death_dttm) as death_date,
        birth_date
    FROM '{patient_path}';
SELECT * FROM patient
""")

# Height

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW height AS
    SELECT
        h.patient_id as mdm_link_id,
        ANY_VALUE(v.vital_value) as height_cm
    FROM '{vitals_path}' v
    INNER JOIN '{hosp_path}' h USING (hospitalizations_joined_id)
    WHERE v.vital_category = 'height_cm'
    GROUP BY mdm_link_id;
SELECT * FROM height
""")

# Hospitalization

In [ ]:
con.sql(f"SELECT * FROM '{hosp_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW hospitalization AS
WITH base AS (
    SELECT
        hospitalizations_joined_id,
        MIN(admission_dttm) AS admission_dttm,
        MAX(discharge_dttm) AS discharge_dttm,
        ANY_VALUE(age_at_admission) AS age,
        ANY_VALUE(elix_vw) AS elix_vw
    FROM '{hosp_path}'
    GROUP BY hospitalizations_joined_id
)
SELECT
    *,
    DATE(admission_dttm) AS admission_date,
    DATE(discharge_dttm) AS discharge_date,
    YEAR(admission_dttm) AS admit_year,
    DATEDIFF('minute', admission_dttm, discharge_dttm) AS hosp_los_min,
    DATEDIFF('minute', admission_dttm, discharge_dttm) / 60.0 AS hosp_los_hr,
    DATEDIFF('minute', admission_dttm, discharge_dttm) / 1440.0 AS hosp_los_day
FROM base;
SELECT * FROM hospitalization
""")

# ICU Type

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE icu_type AS
WITH adt_icu AS (
    SELECT
        hospitalizations_joined_id,
        in_dttm,
        COALESCE(out_dttm, '9999-12-31'::TIMESTAMPTZ) AS out_dttm,
        location_type,
        department_type
    FROM '{adt_path}'
    WHERE location_category = 'icu'
)
SELECT DISTINCT ON (d.hospitalizations_joined_id, d.recorded_dttm)
    d.hospitalizations_joined_id,
    d.recorded_dttm,
    COALESCE(a.location_type,   'Unknown') AS icu_location_type,
    COALESCE(a.department_type, 'Unknown') AS icu_department_type
FROM data d
LEFT JOIN adt_icu a
    ON  d.hospitalizations_joined_id = a.hospitalizations_joined_id
    AND d.recorded_dttm >= a.in_dttm
    AND d.recorded_dttm <  a.out_dttm
ORDER BY d.hospitalizations_joined_id, d.recorded_dttm, a.in_dttm DESC;

SELECT icu_location_type, icu_department_type, COUNT(*) AS n
FROM icu_type
GROUP BY icu_location_type, icu_department_type
ORDER BY n DESC
""")

# ECMO

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW ecmo AS
    SELECT
        REGEXP_REPLACE(hospitalization_id, '_[^_]+$', '') as hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        TRUE as ecmo
    FROM '{ecmo_path}'
    WHERE mcs_group = 'ecmo'
    GROUP BY hospitalizations_joined_id, date;
""")

ecmo_count = con.sql("SELECT COUNT(*) FROM ecmo").fetchone()[0]
if ecmo_count == 0:
    raise ValueError("ecmo view has 0 rows — check ecmo_path and mcs_group filter")
else:
    print(f"ecmo: {ecmo_count} rows")

con.sql("SELECT * FROM ecmo")

# Acidosis (pH < 7.2)

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW ph AS
    SELECT
        hospitalizations_joined_id,
        DATE(lab_collect_dttm) as date,
        MIN(lab_value_numeric) as ph_min_arterial_or_venous
    FROM '{labs_path}'
    WHERE lab_category IN ('ph_arterial', 'ph_venous')
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM ph
""")

# SPO2 daily min

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW spo2_min AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        MIN(vital_value) as spo2_min
    FROM '{vitals_path}'
    WHERE LOWER(vital_category) = 'spo2'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM spo2_min
""")

# PCO2

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW pco2 AS
    SELECT
        hospitalizations_joined_id,
        DATE(lab_collect_dttm) as date,
        MIN(lab_value_numeric) as pco2_min_arterial_or_venous
    FROM '{labs_path}'
    WHERE lower(lab_category) IN ('pco2_venous', 'pco2_arterial')
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM pco2
""")

# SUP

In [ ]:
con.sql(f"SELECT * FROM '{meds_continuous_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW sup AS
WITH continuous AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as sup
    FROM '{meds_continuous_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    )
),
intermittent AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as sup
    FROM '{meds_intermittent_path}'
    WHERE med_category IN (
        'omeprazole',
        'pantoprazole',
        'esomeprazole',
        'lansoprazole',
        'rabeprazole',
        'famotidine',
        'ranitidine',
        'nizatidine',
        'cimetidine'
    )
),
combined AS (
    SELECT * FROM continuous
    UNION ALL
    SELECT * FROM intermittent
)
SELECT *
FROM combined
GROUP BY hospitalizations_joined_id, date, sup;
SELECT * FROM sup
""")

# Paralysis

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW paralytic_meds AS
WITH continuous AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytic_meds
    FROM '{meds_continuous_path}'
    WHERE med_group = 'paralytics'
        AND med_dose != '0'
),
intermittent AS (
    SELECT
        hospitalizations_joined_id,
        DATE(admin_dttm) as date,
        TRUE as paralytic_meds
    FROM '{meds_intermittent_path}'
    WHERE med_group = 'paralytics'
        AND med_dose != '0'
),
combined AS (
    SELECT * FROM continuous
    UNION ALL
    SELECT * FROM intermittent
)
SELECT *
FROM combined
GROUP BY hospitalizations_joined_id, date, paralytic_meds;
SELECT * FROM paralytic_meds
""")

# Vent data daily aggregate

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW vent_daily AS
    SELECT
        rs.hospitalizations_joined_id,
        CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) AS date,
        -- Approximate hours on vent: span of IMV records + 1h buffer
        GREATEST(1, ROUND(
            (EPOCH(MAX(rs.recorded_dttm)) - EPOCH(MIN(rs.recorded_dttm))) / 3600.0 + 1
        )) AS daily_hours_on_vent,
        MIN(rs.fio2_set)  AS vent_fio2_min,
        MAX(rs.fio2_set)  AS vent_fio2_max,
        MIN(rs.peep_set)  AS vent_peep_min,
        MAX(rs.peep_set)  AS vent_peep_max
    FROM '{resp_supp_path}' rs
    INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    GROUP BY rs.hospitalizations_joined_id,
             CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE);
SELECT * FROM vent_daily
""")

# Prone

In [ ]:
con.sql(f"SELECT * FROM '{position_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW prone AS
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) as date,
        TRUE as prone_position
    FROM '{position_path}'
    WHERE position_category = 'prone'
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM prone
""")

# Vent Mode

In [ ]:
con.sql(f"SELECT * FROM '{resp_supp_path}'")

In [ ]:
con.sql(f"SELECT distinct mode_category FROM '{resp_supp_path}'")

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW vent_mode AS
    SELECT
        rs.hospitalizations_joined_id,
        CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) AS date,
        MODE(rs.mode_category) AS primary_mode_category
    FROM '{resp_supp_path}' rs
    INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
    WHERE rs.device_category ILIKE '%imv%'
      AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
              <= cm.ep1_end_local + INTERVAL 2 HOURS
    GROUP BY rs.hospitalizations_joined_id,
             CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE);
SELECT * FROM vent_mode
""")

# Covid

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW covid AS
    SELECT DISTINCT ON (mdm_link_id, date)
        patient_id as mdm_link_id,
        DATE(collect_dttm) as date,
        DATE(collect_dttm) as covid_date,
        TRUE as covid
    FROM '{micro_non_culture_path}'
    WHERE organism_category ILIKE '%sars%'
        AND LOWER(result_category) = 'detected';
SELECT * FROM covid
""")

# Hourly Paired PF Ratio

In [ ]:
po2_arterial = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    DATE(lab_collect_dttm) AS po2_date,
    HOUR(lab_collect_dttm) AS po2_hour,
    lab_collect_dttm AS hour_ts,
    lab_value_numeric AS po2_arterial,
    reference_unit
FROM '{labs_path}'
WHERE lab_category = 'po2_arterial'
""").df()
po2_arterial

In [ ]:
fio2_set_df = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    recorded_date AS date,
    recorded_hour AS hour,
    MAKE_TIMESTAMP(
        YEAR(recorded_date), MONTH(recorded_date), DAY(recorded_date),
        recorded_hour, 0, 0
    ) AS hour_ts,
    vent_episode_id,
    device_category,
    fio2_set
FROM '{hourly_resp_path}'
WHERE device_category = 'imv'
    AND vent_episode_id = '1'
    AND vent_episode_duration_hours >= 24
    AND location_category = 'icu'
""").df()
fio2_set_df

In [ ]:
# merge_asof requires sorted data
fio2_set_df = fio2_set_df.sort_values('hour_ts')
po2_arterial = po2_arterial.sort_values('hour_ts')

po2_arterial['hour_ts'] = po2_arterial['hour_ts'].dt.tz_localize(None)

pf_ratio_df = pd.merge_asof(
    fio2_set_df,
    po2_arterial,
    by='hospitalizations_joined_id',
    on='hour_ts',
    direction='nearest',
    tolerance=pd.Timedelta('1 hour')
)

pf_ratio_df['pf_ratio'] = pf_ratio_df['po2_arterial'] / pf_ratio_df['fio2_set']
pf_ratio_df = pf_ratio_df[pf_ratio_df['pf_ratio'].notna()]

pf_ratio_df

In [ ]:
con.register('pf_ratio_df', pf_ratio_df)
con.sql(f"""
CREATE OR REPLACE TEMP VIEW pf_ratio_paired AS
    SELECT
        hospitalizations_joined_id,
        date(date) as date,
        MIN(pf_ratio) AS pf_ratio_paired_min
    FROM pf_ratio_df
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM pf_ratio_paired
""")

In [ ]:
# SF ratio: pair SpO2 (from vitals) with FiO2 (from raw resp_supp),
# computed like pf_ratio_paired — nearest SpO2 within 1h of each raw FiO2 record.
# Both timestamps converted to naive local time before merge_asof (consistent
# with the pf_ratio approach above).

fio2_raw_df = con.sql(f"""
SELECT
    rs.hospitalizations_joined_id,
    CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP) AS fio2_dttm,
    CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE)      AS date,
    rs.fio2_set
FROM '{resp_supp_path}' rs
INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
WHERE rs.device_category ILIKE '%imv%'
  AND rs.fio2_set IS NOT NULL
  AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
          <= cm.ep1_end_local + INTERVAL 2 HOURS
""").df()

spo2_df = con.sql(f"""
SELECT
    hospitalizations_joined_id,
    CAST(recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP) AS spo2_dttm,
    vital_value AS spo2
FROM '{vitals_path}'
WHERE vital_category = 'spo2'
  AND vital_value IS NOT NULL
""").df()

fio2_raw_df = fio2_raw_df.sort_values('fio2_dttm')
spo2_df     = spo2_df.sort_values('spo2_dttm')

sf_ratio_raw = pd.merge_asof(
    fio2_raw_df,
    spo2_df,
    left_on='fio2_dttm',
    right_on='spo2_dttm',
    by='hospitalizations_joined_id',
    direction='nearest',
    tolerance=pd.Timedelta('1 hour')
)

sf_ratio_raw['sf_ratio'] = sf_ratio_raw['spo2'] / sf_ratio_raw['fio2_set']
sf_ratio_raw = sf_ratio_raw[sf_ratio_raw['sf_ratio'].notna()]

con.register('sf_ratio_raw', sf_ratio_raw)
con.sql("""
CREATE OR REPLACE TEMP VIEW sf_ratio_paired AS
    SELECT
        hospitalizations_joined_id,
        date,
        MIN(sf_ratio) AS sf_ratio
    FROM sf_ratio_raw
    GROUP BY hospitalizations_joined_id, date;
SELECT * FROM sf_ratio_paired
""")

# LAPS2

## Save vent hospitalizations for LAPS2 export via R function

In [ ]:
con.sql(f"""
    COPY (
        SELECT
            hospitalizations_joined_id,
            ANY_VALUE(mdm_link_id) AS mdm_link_id,
            ANY_VALUE(mdm_link_id) AS patient_id
        FROM data
        GROUP BY hospitalizations_joined_id
    ) TO 'laps2_hospitalizations.parquet' (FORMAT PARQUET)
""")

## Call laps2 function and create laps2 view

In [ ]:
# !run_laps2.bat
# shutil.move('laps2_data.parquet', f'{output_path}/laps2_data.parquet')
# shutil.move('laps2_hospitalizations.parquet', f'{output_path}/laps2_hospitalizations.parquet')
# shutil.move('status.jsonl', f'{output_path}/status.jsonl')

In [ ]:
con.sql(f"""
CREATE OR REPLACE TEMP VIEW laps2 AS
    SELECT DISTINCT ON (hospitalizations_joined_id, recorded_date)
        hospitalizations_joined_id,
        recorded_date as date,
        laps2
    FROM '{output_path}/laps2_data.parquet';
SELECT * FROM laps2
""")

# Merge duckdb tables

In [ ]:
print(con.sql(f"SELECT COUNT(*) FROM data"))
con.sql(f"""
CREATE OR REPLACE TABLE final_df AS
    SELECT
        data.*,
        patient.* EXCLUDE (mdm_link_id),
        height.height_cm,
        hospitalization.* EXCLUDE (hospitalizations_joined_id),
        vent_daily.* EXCLUDE (hospitalizations_joined_id, date),
        vent_mode.primary_mode_category,
        ph.ph_min_arterial_or_venous   AS ph,
        pco2.pco2_min_arterial_or_venous AS pco2,
        spo2_min.spo2_min,
        sf_ratio_paired.sf_ratio,
        prone.prone_position,
        sup.sup,
        ecmo.ecmo,
        paralytic_meds.paralytic_meds,
        laps2.laps2,
        pf_ratio_paired.pf_ratio_paired_min,
        icu_type.icu_location_type,
        icu_type.icu_department_type
    FROM data
    LEFT JOIN patient         USING (mdm_link_id)
    LEFT JOIN height          USING (mdm_link_id)
    LEFT JOIN hospitalization USING (hospitalizations_joined_id)
    LEFT JOIN vent_daily      USING (hospitalizations_joined_id, date)
    LEFT JOIN vent_mode       USING (hospitalizations_joined_id, date)
    LEFT JOIN ph              USING (hospitalizations_joined_id, date)
    LEFT JOIN pco2            USING (hospitalizations_joined_id, date)
    LEFT JOIN spo2_min        USING (hospitalizations_joined_id, date)
    LEFT JOIN sf_ratio_paired USING (hospitalizations_joined_id, date)
    LEFT JOIN prone           USING (hospitalizations_joined_id, date)
    LEFT JOIN sup             USING (hospitalizations_joined_id, date)
    LEFT JOIN ecmo            USING (hospitalizations_joined_id, date)
    LEFT JOIN paralytic_meds  USING (hospitalizations_joined_id, date)
    LEFT JOIN laps2           USING (hospitalizations_joined_id, date)
    LEFT JOIN pf_ratio_paired USING (hospitalizations_joined_id, date)
    LEFT JOIN icu_type       USING (hospitalizations_joined_id, recorded_dttm);
SELECT * FROM final_df
""")
print(con.sql(f"SELECT COUNT(*) FROM final_df"))
# Row counts must match

In [ ]:
data = con.sql(f"SELECT * FROM final_df ORDER BY hospitalizations_joined_id, date").df()
data

In [ ]:
# ICU composite for Task 3: icu_location_type × icu_department_type
data['icu_composite'] = data['icu_location_type'] + ' - ' + data['icu_department_type']
print(data['icu_composite'].value_counts())

In [ ]:
# Task 22 — IBW NULL audit (pre-recovery baseline)
# Note: includes hospitals later excluded at cell 81 (hospital_id == 'delete')
ibw_null = data[data['ibw'].isna()].copy()
ibw_null_hosp_n = ibw_null['hospitalizations_joined_id'].nunique()
ibw_null_days_n = len(ibw_null)

print(f"Patient-days with NULL ibw: {ibw_null_days_n:,}")
print(f"Unique hospitalizations with NULL ibw: {ibw_null_hosp_n:,}")
print(f"  ({ibw_null_hosp_n / data['hospitalizations_joined_id'].nunique() * 100:.1f}% of total hospitalizations)\n")

print("By hospital_id:")
print(ibw_null.groupby('hospital_id')['hospitalizations_joined_id'].nunique().to_string())

print("\nBy year:")
print(
    ibw_null.assign(year=pd.to_datetime(ibw_null['date'], errors='coerce').dt.year)
    .groupby('year')['hospitalizations_joined_id'].nunique()
    .to_string()
)

In [ ]:
# Task 22 — Derive IBW from height_cm + sex_category (Devine formula) for NULL-IBW rows
# Plausible height range per ards_classifier.R: 76.2–244 cm
# height_cm is patient-level (pooled across all hospitalizations via mdm_link_id); benign
# for adults since height is stable across admissions.
HEIGHT_MIN_CM, HEIGHT_MAX_CM = 76.2, 244.0

# Ensure float dtype before assignment (guards against int64 from DuckDB)
data['ibw'] = data['ibw'].astype(float)

# Snapshot before recovery — both row and hospitalization counts
n_days_before = int(data['ibw'].isna().sum())
n_hosp_before = int(data.loc[data['ibw'].isna(), 'hospitalizations_joined_id'].nunique())

recovery_mask = (
    data['ibw'].isna()
    & data['height_cm'].notna()
    & data['height_cm'].between(HEIGHT_MIN_CM, HEIGHT_MAX_CM)
    & data['sex_category'].isin(['Male', 'Female'])
)

# Devine formula; clip to base weight for patients shorter than 5 ft (60 in)
male_mask   = recovery_mask & (data['sex_category'] == 'Male')
female_mask = recovery_mask & (data['sex_category'] == 'Female')

data.loc[male_mask,   'ibw'] = (50   + 2.3 * (data.loc[male_mask,   'height_cm'] / 2.54 - 60)).clip(lower=50.0)
data.loc[female_mask, 'ibw'] = (45.5 + 2.3 * (data.loc[female_mask, 'height_cm'] / 2.54 - 60)).clip(lower=45.5)

# Snapshot after recovery
n_days_after     = int(data['ibw'].isna().sum())
n_hosp_after     = int(data.loc[data['ibw'].isna(), 'hospitalizations_joined_id'].nunique())
n_days_recovered = n_days_before - n_days_after
n_hosp_recovered = int(data.loc[recovery_mask, 'hospitalizations_joined_id'].nunique())

print(f"NULL ibw before recovery  — patient-days: {n_days_before:,}  |  hospitalizations: {n_hosp_before:,}")
print(f"Recovered via Devine      — patient-days: {n_days_recovered:,}  |  hospitalizations: {n_hosp_recovered:,}")
print(f"Still NULL after recovery — patient-days: {n_days_after:,}  |  hospitalizations: {n_hosp_after:,}  (excluded at cell 81)")

# BMI

In [ ]:
weight = con.sql(f"""
    SELECT
        hospitalizations_joined_id,
        DATE(recorded_dttm) AS date,
        vital_value AS weight_kg
    FROM '{vitals_path}'
    WHERE vital_category = 'weight_kg'
        AND vital_value IS NOT NULL
    ORDER BY recorded_dttm
""").df()

data = pd.merge_asof(
    data.sort_values('date'),
    weight.sort_values('date'),
    by='hospitalizations_joined_id',
    on='date',
    direction='nearest',
    allow_exact_matches=True
)

data = data.sort_values(by=['hospitalizations_joined_id', 'date'])
data['bmi_calc'] = round(data['weight_kg'] / (data['height_cm'] / 100) ** 2, 2)
data

In [ ]:
# when is tidal_obs and not tidal_vol set > that is the non-volume control number

In [ ]:
data[['hospitalizations_joined_id', 'date', 'tidal_volume_obs', 'tidal_volume_set']]

## Merge in covid data within 2 weeks from first patient day. Carry forward?

In [ ]:
df = con.sql("""
             SELECT *
             FROM covid
             ORDER BY date
""").df().dropna()

intubation = data.sort_values(by='date').drop_duplicates(subset='hospitalizations_joined_id', keep='first')

# Merge within 14 days of intubation
intubation = pd.merge_asof(
    intubation,
    df,
    by='mdm_link_id',
    on='date',
    direction='backward',
    tolerance=pd.Timedelta('14D'),
    allow_exact_matches=True
)
intubation = intubation.sort_values(by=['hospitalizations_joined_id', 'date'])
print(f'Hospitalizations positive for covid: {len(intubation[intubation['covid'] == True])}')

intubation[intubation['covid'] == True][['hospitalizations_joined_id', 'mdm_link_id', 'date', 'covid_date', 'covid']]

In [ ]:
# Merge back onto main dataframe
data = data.merge(intubation[['hospitalizations_joined_id', 'covid']], on='hospitalizations_joined_id', how='left')
data[data['covid'] == True]

# Pandas engineering

In [ ]:
# Map hospital IDs
hosp_ids = {
    'Woodwinds' : 'Hospital 1',
    'St. Johns' : 'Hospital 2',
    'Southdale' : 'Hospital 3',
    'Riverside' : 'Hospital 4',
    'Ridges' : 'Hospital 5',
    'Ranges/Hibbing' : 'Hospital 6',
    'Northland' : 'Hospital 7',
    'Lakes' : 'Hospital 8',
    'Grand Itasca' : 'delete',
    'UMMC' : 'Hospital Ref',
    'St. Joes' : 'delete',
    'Bethesda' : 'delete'
}
data['hospital_id'] = data['hospital_id'].map(hosp_ids)
data[['hospital_id']]

In [ ]:
# Get mortality
data['mortality_inhospital'] = (data['death_date'].dt.date <= data['discharge_dttm'].dt.date).astype(bool)

In [ ]:
# Check for vent duration of at least 48hr for sup model
data['vent_episode_duration_48hr_min'] = (data['vent_episode_duration_hours'] >= 48).astype(bool)

In [ ]:
# Fillna meds / prone / sup
data['paralytic_meds'] = data['paralytic_meds'].fillna(False)
data['prone_position'] = data['prone_position'].fillna(False)
data['sup'] = data['sup'].fillna(False)

# COVID - fill in False
data['covid'] = data['covid'].fillna(False)

In [ ]:
# LTVV outcomes — derived from raw tidal_volume_set (Task 1).
# Rows where tidal_volume_set is NaN (no raw Vt found at correct timepoint)
# will compute as 0 here; Cell 82 drops those rows before the final parquet.

# Primary outcomes: raw set Vt only
data['tidal_volume_set_ibw'] = data['tidal_volume_set'] / data['ibw']
data['ltvv_8'] = (data['tidal_volume_set_ibw'] <= 8).astype(int)
data['ltvv_6'] = (data['tidal_volume_set_ibw'] <= 6).astype(int)

# Secondary fallback outcomes: raw set-or-obs
data['tidal_volume_set_or_obs'] = data['tidal_volume_set'].fillna(data['tidal_volume_obs'])
data['tidal_volume_set_or_obs_ibw'] = data['tidal_volume_set_or_obs'] / data['ibw']
data['ltvv_tidal_volume_set_or_obs_8'] = (data['tidal_volume_set_or_obs_ibw'] <= 8).astype(int)
data['ltvv_tidal_volume_set_or_obs_6'] = (data['tidal_volume_set_or_obs_ibw'] <= 6).astype(int)

print(f"ltvv_6 rate (pre-exclusion): {data['ltvv_6'].mean()*100:.1f}%")
print(f"ltvv_8 rate (pre-exclusion): {data['ltvv_8'].mean()*100:.1f}%")
data.head(5)

## Carry forward sf_ratio min, pf ratio min, ph min, and pco2 max up to 3 days

In [ ]:
# Carry forward sf_ratio, ph, pco2, and pf_ratio_paired_min up to 3 days max
data = data.sort_values(by=['hospitalizations_joined_id', 'date'])

data['ph'] = (
    data.groupby('hospitalizations_joined_id')['ph']
    .ffill(limit=3)
)
data['pco2'] = (
    data.groupby('hospitalizations_joined_id')['pco2']
    .ffill(limit=3)
)
data['sf_ratio'] = (
    data.groupby('hospitalizations_joined_id')['sf_ratio']
    .ffill(limit=3)
)

# Carry forward pf_ratio_paired_min up to 3 days max
data['pf_ratio_paired_min'] = (
    data.groupby('hospitalizations_joined_id')['pf_ratio_paired_min']
    .ffill(limit=3)
)

data

# Exclusion/Inclusion

In [ ]:
# Remove tracheostomy rows
print(f'Rows before dropping trach: {len(data)}')
data = data[data['tracheostomy'] != 1].copy()
data['tracheostomy'] = data['tracheostomy'].fillna(False).astype(bool)
print(f'Rows after dropping trach: {len(data)}')

data

In [ ]:
# Remove ECMO days
print(len(data))
data['ecmo'] = data['ecmo'].fillna(False)
data = data[data['ecmo'] == False].copy()
print(len(data))

In [ ]:
## Leftover exclusions
# Make sure 18+
print(f'Starting row n: {len(data)}')
data = data[data['age'] >= 18]
print(f'Age: {len(data)}')

# Remove st. joes, bethesda, and grand itasca
data = data[data['hospital_id'] != 'delete']
print(f'Hospital ID: {len(data)}')

# Require non-null IBW; recovery from height_cm was attempted above (Task 22)
data = data[data['ibw'].notna()]
print(f'IBW not null (post-recovery): {len(data)}')

# Remove weird stuff
data = data.dropna(subset=['prov_npi_shifted'])
data = data[~data['prov_npi_shifted'].isin(['None', 'none', 'nan'])]
print(f'Provider: {len(data)}')

data

In [ ]:
# Keep only if tidal_volume_set not null
print(f"Before dropping na: {len(data)}")
data = data[~data['tidal_volume_set'].isna()]
print(f"After dropping na: {len(data)}")

data

# Missingness

In [ ]:
print(len(data))
data.isna().sum()

# AHRF

In [ ]:
# !run_ahrf.bat
# shutil.move('ahrf_classifier_cohort.csv', f'{output_path}/ahrf_classifier_cohort.csv')
# shutil.move('status.jsonl', f'{output_path}/status.jsonl')
# shutil.move('all_hosp_ids_on_vent.parquet', f'{output_path}/all_hosp_ids_on_vent.parquet')

In [ ]:
ahrf_classifier_cohort = pd.read_csv(f'{output_path}/ahrf_classifier_cohort.csv', index_col=0)
ahrf_classifier_cohort["hospitalizations_joined_id"] = ahrf_classifier_cohort["hospitalization_id"].str.replace(
    r"^([^_]+_[^_]+).*", r"\1", regex=True)
print(len(ahrf_classifier_cohort))

# some duplicate rows remain - drop
ahrf_classifier_cohort = ahrf_classifier_cohort.drop_duplicates()
print(len(ahrf_classifier_cohort))

# Some duplicate hospitalizations_joined_id > only difference is cardiac_arrest_primary_dx column
# Remove duplicate cardiac_arres_primary_dx
ahrf_classifier_cohort = ahrf_classifier_cohort.sort_values('cardiac_arrest_primary_dx', ascending = False)\
    .drop_duplicates('hospitalizations_joined_id')
print(len(ahrf_classifier_cohort))

# Merge onto data
print(len(data))
data = data.merge(ahrf_classifier_cohort[['hospitalizations_joined_id', 'cohort_eligible']], on='hospitalizations_joined_id', how='inner')
data['ahrf_eligible'] = data['cohort_eligible']
data

# Missingness Table (Task 9)

In [ ]:
# Model variables and their missingness handling methods
miss_vars = {
    'ltvv_6':            ('LTVV adherence (≤6 mL/kg IBW)',              'Outcome — complete'),
    'age':               ('Age (years)',                                  'MICE PMM'),
    'sex_category':      ('Sex',                                          'MICE PMM'),
    'race_category':     ('Race/ethnicity',                               'Complete case'),
    'height_cm':         ('Height (cm)',                                  'MICE PMM'),
    'bmi_calc':          ('BMI (kg/m²)',                             'MICE PMM'),
    'elix_vw':           ('Elixhauser comorbidity score',                 'MICE PMM'),
    'laps2':             ('LAPS2 severity score',                         'MICE PMM'),
    'ph':                ('pH (arterial or venous)',                      'MICE PMM; 3-day carry-forward'),
    'pco2':              ('pCO2 (arterial or venous)',                    'MICE PMM; 3-day carry-forward'),
    'sf_ratio':          ('SF ratio (SpO2/FiO2)',                         'MICE PMM; 3-day carry-forward'),
    'hospital_id':       ('Hospital',                                     'MICE PMM'),
    'icu_location_type': ('ICU location type',                            'Complete (COALESCE → Unknown)'),
    'recorded_year':     ('Calendar year',                                'Always present'),
    'prov_npi_shifted':  ('Attending provider',                           'Always present (required)'),
}

n_total = len(data)
rows = []
for col, (label, method) in miss_vars.items():
    if col not in data.columns:
        raise ValueError(f"Column '{col}' not found in data — check column name")
    n_miss = int(data[col].isna().sum())
    pct = n_miss / n_total * 100
    rows.append({
        'Variable': label,
        'Column': col,
        'N': n_total,
        'N Missing': n_miss,
        '% Missing': round(pct, 1),
        'Missing Data Handling': method,
    })

miss_df = pd.DataFrame(rows)
miss_df.to_excel(f'{output_path}/etable_task9_missingness.xlsx', index=False)
display(miss_df[['Variable', 'N Missing', '% Missing', 'Missing Data Handling']])

high_miss = miss_df[miss_df['% Missing'] > 20]
if not high_miss.empty:
    print(f"\nWARNING: {len(high_miss)} variable(s) exceed 20% missingness:")
    print(high_miss[['Variable', '% Missing']].to_string(index=False))
else:
    print("\nNo variables exceed 20% missingness threshold.")

# Figure: Within-Day Tidal Volume Distribution (Task 8)

In [ ]:
# All IMV tidal_volume_set records per (hospitalization, local_date) within episode 1
vt_all = con.sql(f"""
SELECT
    rs.hospitalizations_joined_id,
    CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS DATE) AS date,
    rs.tidal_volume_set / cm.ibw                              AS vt_ibw
FROM '{resp_supp_path}' rs
INNER JOIN cohort_meta cm USING (hospitalizations_joined_id)
WHERE rs.device_category ILIKE '%imv%'
  AND rs.tidal_volume_set IS NOT NULL
  AND cm.ibw IS NOT NULL
  AND CAST(rs.recorded_dttm AT TIME ZONE '{timezone}' AS TIMESTAMP)
          <= cm.ep1_end_local + INTERVAL 2 HOURS
""").df()

vt_daily_stats = (
    vt_all
    .groupby(['hospitalizations_joined_id', 'date'])['vt_ibw']
    .agg(n='count', mean='mean', std='std', vt_min='min', vt_max='max')
    .reset_index()
)
vt_daily_stats['cv']    = (vt_daily_stats['std'] / vt_daily_stats['mean']) * 100
vt_daily_stats['range'] = vt_daily_stats['vt_max'] - vt_daily_stats['vt_min']
vt_multi = vt_daily_stats[vt_daily_stats['n'] >= 2].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(vt_multi['cv'].clip(upper=50), bins=50, edgecolor='none', color='steelblue')
ax1.axvline(x=10, linestyle='--', color='red', label='CV = 10%')
ax1.set_xlabel('Within-Day CV of Vt/kg IBW (%)')
ax1.set_ylabel('Number of Patient-Days')
ax1.set_title('A. Within-Day Vt Variability')
ax1.legend()
pct_low_cv = (vt_multi['cv'] < 10).mean() * 100
ax1.text(0.95, 0.95, f'{pct_low_cv:.1f}% of days\nhave CV < 10%',
         transform=ax1.transAxes, ha='right', va='top', fontsize=11)

ax2.hist(data['tidal_volume_set_ibw'].dropna().clip(lower=2, upper=20),
         bins=60, edgecolor='none', color='steelblue')
ax2.axvline(x=6, linestyle='--', color='red', linewidth=2, label='6 mL/kg PBW')
ax2.axvline(x=8, linestyle=':', color='orange', linewidth=1.5, label='8 mL/kg PBW')
ax2.set_xlabel('Representative Vt/kg IBW (mL/kg)')
ax2.set_ylabel('Number of Patient-Days')
ax2.set_title('B. Distribution of Set Tidal Volume')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{output_path}/fig_task8_vt_distribution.tiff', dpi=600, bbox_inches='tight')
plt.savefig(f'{output_path}/fig_task8_vt_distribution.png',  dpi=150, bbox_inches='tight')
plt.show()

print(f"n patient-days with >=2 Vt records: {len(vt_multi):,}")
print(f"Median within-day CV: {vt_multi['cv'].median():.2f}% [IQR {vt_multi['cv'].quantile(0.25):.2f}–{vt_multi['cv'].quantile(0.75):.2f}%]")
print(f"Median within-day range: {vt_multi['range'].median():.3f} [IQR {vt_multi['range'].quantile(0.25):.3f}–{vt_multi['range'].quantile(0.75):.3f}] mL/kg")
pct_cv_low  = (vt_multi['cv'] < 10).mean() * 100
print(f"% days with CV < 10%: {pct_cv_low:.1f}%")

# Save final dataframe

## Table 1

### Definition to create table 1

In [ ]:
def create_table1(df, continuous_vars, categorical_vars, var_order=None):
    """
    Create a Table 1 summary statistics table

    Parameters
    ----------
    df : pandas DataFrame
        The patient data
    continuous_vars : dict
        {column_name: display_label} for continuous vars
        e.g., {'age': 'Age, median [Q1,Q3]'}
    categorical_vars : dict
        {column_name: display_label} for categorical vars
        e.g., {'race': 'Race, n (%)'}
    var_order : list | None
        Exact row order; use 'n' for the total count.
        Example: ['n', 'age', 'bmi_calc', 'race_category', 'sex_category', ...]

    Returns
    -------
    pandas DataFrame with columns ['Variable','Category','Value']
    """

    results = []

    # Total N
    total_n = len(df)

    def _add_n():
        results.append({'Variable': 'n', 'Category': '', 'Value': str(total_n)})

    def _add_cont(var):
        if var in df.columns:
            label = continuous_vars[var]
            median = df[var].median()
            q1 = df[var].quantile(0.25)
            q3 = df[var].quantile(0.75)
            value = f"{median:.1f} [{q1:.1f},{q3:.1f}]"
            results.append({'Variable': label, 'Category': '', 'Value': value})

    def _add_cat(var):
        if var in df.columns:
            label = categorical_vars[var]
            # header row
            results.append({'Variable': label, 'Category': '', 'Value': ''})
            # category rows
            value_counts = df[var].value_counts()
            for category, count in value_counts.items():
                pct = (count / total_n) * 100
                value = f"{count} ({pct:.1f})"
                results.append({'Variable': '', 'Category': str(category), 'Value': value})

    if var_order is not None:
        # Follow the exact order provided
        for key in var_order:
            if key == 'n':
                _add_n()
            elif key in continuous_vars:
                _add_cont(key)
            elif key in categorical_vars:
                _add_cat(key)
            # else: silently ignore unknown keys
    else:
        # Original behavior (keep as-is)
        _add_n()
        for var, _label in continuous_vars.items():
            _add_cont(var)
        for var, _label in categorical_vars.items():
            _add_cat(var)

    return pd.DataFrame(results)

### Create table 1

In [ ]:
table1_data = data.groupby(['hospitalizations_joined_id']).agg(
    age=('age', 'first'),
    bmi_calc=('bmi_calc', 'median'),
    elix_vw=('elix_vw', 'median'),
    pf_ratio_paired_min=('pf_ratio_paired_min', 'median'),
    pco2=('pco2', 'median'),
    ph=('ph', 'median'),
    laps2=('laps2', 'median'),
    vent_episode_duration_hours=('vent_episode_duration_hours', 'first'),
    hosp_los_day=('hosp_los_day', 'first'),
    race_category=('race_category', 'first'),
    sex_category=('sex_category', 'first'),
    admit_year=('admit_year', 'first'),
    hospital_id=('hospital_id', 'first'),
    mortality_inhospital=('mortality_inhospital', 'first'),
    ahrf_eligible=('ahrf_eligible', 'first'),
    icu_location_type=('icu_location_type', lambda x: x.mode()[0] if not x.mode().empty else None)
).reset_index()

table1_data

In [ ]:
# Define variables for Table 1
continuous_vars = {
    'age': 'Age, median [Q1,Q3]',
    'bmi_calc': 'BMI, median [Q1,Q3]',
    'elix_vw': 'Elixhauser, median [Q1,Q3]',
    'pf_ratio_paired_min': 'PF Ratio (paired), median [Q1,Q3]',
    'pco2': 'pCO2 (arterial or venous), median [Q1,Q3]',
    'ph': 'pH (arterial or venous), median [Q1,Q3]',
    'laps2': 'Laps2, median [Q1,Q3]',
    'vent_episode_duration_hours': 'Vent episode duration (hr), median [Q1,Q3]',
    'hosp_los_day': 'Hospitalization length (days), median [Q1,Q3]'
}

categorical_vars = {
    'race_category': 'Race, n (%)',
    'sex_category': 'Sex, n (%)',
    'admit_year': 'Year, n (%)',
    'hospital_id': 'Hospital, n (%)',
    'mortality_inhospital': 'In-hospitalization mortality, n (%)',
    'icu_location_type': 'ICU Location Type, n (%)'
}

var_order = ['n',
             'age', 'race_category', 'sex_category', 'icu_location_type',
             'bmi_calc', 'elix_vw', 'laps2',
             'pf_ratio_paired_min', 'pco2', 'ph', 'vent_episode_duration_hours',
             'hosp_los_day', 'mortality_inhospital', 'hospital_id', 'admit_year'
]

# Generate Table 1 (overall only)
table1 = create_table1(table1_data, continuous_vars, categorical_vars, var_order)

# Generate Table 1 stratified by AHRF cohort eligibility
table1_stratified = table1.rename(columns={'Value': 'Overall'}).copy()

for cohort_value in sorted(table1_data['ahrf_eligible'].dropna().unique()):
    cohort_df = table1_data[table1_data['ahrf_eligible'] == cohort_value]
    if cohort_df.empty:
        continue

    if cohort_value in (1, True):
        cohort_label = 'Persistent AHRF cohort'
    elif cohort_value in (0, False):
        cohort_label = 'Non-AHRF cohort'
    else:
        cohort_label = f'ahrf_eligible={cohort_value}'

    cohort_table = create_table1(cohort_df, continuous_vars, categorical_vars, var_order)
    cohort_table = cohort_table.rename(columns={'Value': cohort_label})
    table1_stratified = table1_stratified.merge(
        cohort_table[['Variable', 'Category', cohort_label]],
        on=['Variable', 'Category'],
        how='left'
    )

# Task 15: append patients-per-provider (median [Q1-Q3]) for each sub-cohort
def _ppp_stat(df_subset):
    sub = df_subset.drop_duplicates(subset=['hospitalizations_joined_id', 'prov_npi_shifted'])
    counts = sub['prov_npi_shifted'].value_counts()
    return f"{round(counts.median())} [{round(counts.quantile(0.25))}–{round(counts.quantile(0.75))}]"

ppp_label = 'Patients per provider, median [Q1, Q3]'
table1 = pd.concat([table1, pd.DataFrame([{
    'Variable': ppp_label, 'Category': '', 'Value': _ppp_stat(data)
}])], ignore_index=True)
table1_stratified = pd.concat([table1_stratified, pd.DataFrame([{
    'Variable':                ppp_label,
    'Category':                '',
    'Overall':                 _ppp_stat(data),
    'Persistent AHRF cohort':  _ppp_stat(data[data['ahrf_eligible'] == 1]),
    'Non-AHRF cohort':         _ppp_stat(data[data['ahrf_eligible'] == 0]),
}])], ignore_index=True)

table1.to_excel(f'{output_path}/table1.xlsx', index=False)
table1_stratified.to_excel(f'{output_path}/table1_stratified_by_ahrf_eligible.xlsx', index=False)

table1, table1_stratified

# Provider counts

In [ ]:
# Task 15: patients-per-provider by sub-cohort (median, IQR)
# _ppp_stat defined in cell id=96; reused here
subcohorts = {
    'Overall':                data,
    'Persistent AHRF':        data[data['ahrf_eligible'] == 1],
    'Non-AHRF':               data[data['ahrf_eligible'] == 0],
    'Day-1 stratum':          data[data['initial_vent_day']],
    'Subsequent-day stratum': data[~data['initial_vent_day']],
}

rows = []
for label, df_sub in subcohorts.items():
    sub = df_sub.drop_duplicates(subset=['hospitalizations_joined_id', 'prov_npi_shifted'])
    counts = sub['prov_npi_shifted'].value_counts()
    rows.append({
        'Sub-cohort':   label,
        'N providers':  len(counts),
        'Median':       counts.median(),
        'Q1':           counts.quantile(0.25),
        'Q3':           counts.quantile(0.75),
        'Formatted':    _ppp_stat(df_sub),
    })

ppp_df = pd.DataFrame(rows)
print('Patients per provider by sub-cohort (Task 15):')
display(ppp_df)

# Save final data to parquet

In [ ]:
data.to_parquet(f'{output_path}/LPV_final_data.parquet')
print(f'Hospitalization Number: {len(data['hospitalizations_joined_id'].unique())}')
print(f'Patient Number: {len(data['mdm_link_id'].unique())}')
print(f'Provider Number: {len(data['prov_npi_shifted'].unique())}')
print(f'Date Number: {len(data)}')

In [ ]:
ahrf_data = data[data['ahrf_eligible'] == 1]
print(f'Hospitalization Number: {len(ahrf_data['hospitalizations_joined_id'].unique())}')
print(f'Patient Number: {len(ahrf_data['mdm_link_id'].unique())}')
print(f'Provider Number: {len(ahrf_data['prov_npi_shifted'].unique())}')
print(f'Date Number: {len(ahrf_data)}')
ahrf_data